"""
========================================================================
NOTEBOOK 05: QUALITATIVE CODING (LLM-as-sociological-reader)
========================================================================

Purpose: capture the dimensions of NYT vs LLM-generated article
difference that DistilBERT/VADER/lexicon analysis misses.

Design:
  - Strong model (GPT-4o) reads paired articles (NYT + GPT-4o-mini, same
    headline, V.A rep 0) and produces a structured observational text.
  - For each pair, GPT-4o is asked to comment on:
    * Writing register and structural choices
    * Stance / framing / what's foregrounded
    * Sociological texture (who speaks, who has agency, who is absent)
    * Political slant cues (explicit and implicit)
    * What 'NYT-ness' is missing or different
  - Output: structured JSON with ratings + a free-form qualitative pass
  - 20 pairs total. ~$1-3 cost. ~5-10 minutes runtime.

Inputs:
  data/processed/experiment_sample.csv  (NYT real text)
  data/generations/generations_gpt.csv  (GPT-4o-mini generations)

Outputs:
  data/results/qualitative_coding_raw.csv      raw GPT-4o judgments
  data/results/qualitative_coding_aggregated.json  themes summary
"""

In [ ]:
# Set your OpenAI API key in the environment BEFORE running (do NOT hardcode):
#   export OPENAI_API_KEY="sk-..."
import os
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY in your environment first."

In [ ]:

# =============================================================================
# CELL 1: Setup
# =============================================================================

import os
import json
import time
from pathlib import Path

import pandas as pd
from openai import OpenAI

ROOT = Path("/Users/apple/Downloads/cultureNYT_clean").resolve()
PROCESSED = ROOT / "data" / "processed"
GENERATIONS = ROOT / "data" / "generations"
RESULTS = ROOT / "data" / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

assert os.environ.get("OPENAI_API_KEY"), \
    "OPENAI_API_KEY not set. Run: os.environ['OPENAI_API_KEY'] = 'sk-...'"

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# Use GPT-4o (full) — we need real reading-quality judgment
JUDGE_MODEL = "gpt-4o"
TEMPERATURE = 0.3   # low temperature for consistent judgments
MAX_TOKENS = 2500


In [ ]:

# =============================================================================
# CELL 2: Load paired articles
# =============================================================================

sampled = pd.read_csv(PROCESSED / "experiment_sample.csv")
df_gen = pd.read_csv(GENERATIONS / "generations_gpt.csv")

# Pair: NYT (sample_id=N, full_text) + GPT-4o-mini A rep=0 (sample_id=N)
pairs = []
for _, nyt_row in sampled.iterrows():
    sid = nyt_row["sample_id"]
    gen_match = df_gen[
        (df_gen["sample_id"] == sid)
        & (df_gen["prompt_version"] == "A")
        & (df_gen["rep"] == 0)
    ]
    if len(gen_match) == 0:
        print(f"  ! No V.A rep=0 generation for sample_id={sid}, skipping")
        continue
    gen_row = gen_match.iloc[0]
    pairs.append({
        "sample_id": sid,
        "headline": nyt_row["headline"],
        "news_desk": nyt_row["news_desk"],
        "pub_date": str(nyt_row["pub_date"]),
        "nyt_text": nyt_row["full_text"],
        "gpt_text": gen_row["generated_full_text"],
    })

pairs_df = pd.DataFrame(pairs)
print(f"Loaded {len(pairs_df)} paired articles")
print(pairs_df.groupby("news_desk").size())

In [ ]:

# =============================================================================
# CELL 3: Define the qualitative coding prompt
# =============================================================================

JUDGE_SYSTEM = """You are a careful reader trained in sociology of media, journalism studies, and discourse analysis. You will be shown two news articles on the same topic: one is the original New York Times article (Article A), and one is a generated article (Article B) produced by an AI model from the same headline, date, and section label.

Your job is to read both articles carefully and produce a fine-grained comparative analysis. You are not asked to judge factual accuracy. You are asked to identify the differences a sociologically-trained reader would notice — differences in writing register, framing, stance, sociological texture, and political cues — that go beyond simple sentiment or word choice.

Read both articles in full before responding. Be specific: cite particular sentences or phrasings when you make a claim. Be honest: if the two articles are similar on a dimension, say so."""


JUDGE_USER_TEMPLATE = """## Article A (Real NYT, {desk} desk, {date})
Headline: {headline}

{nyt_text}

---

## Article B (AI-generated, same headline/date/desk)

{gpt_text}

---

## Your task

Compare Article B to Article A on the dimensions below. For each dimension, write 2-4 sentences. Quote specific phrases when relevant. Then provide an overall summary.

Respond in this exact JSON format:

{{
  "writing_register": {{
    "observation": "How does Article B's prose register differ from A? Consider: sentence rhythm, paragraph length, how openings work, use of concrete date markers vs abstract framings, attribution style.",
    "B_more_or_less_NYT_like": "more|less|similar"
  }},
  "stance_and_framing": {{
    "observation": "What does each article foreground and background? What is treated as the central tension or stakes? Where does each place its critical edge, if any?",
    "B_more_or_less_NYT_like": "more|less|similar"
  }},
  "sociological_texture": {{
    "observation": "Who speaks in each article? Who has agency and who is acted upon? Whose perspective is centered? Who is absent or only mentioned in passing? Are sources named individuals or generic categories ('analysts', 'experts')?",
    "B_more_or_less_NYT_like": "more|less|similar"
  }},
  "political_slant_cues": {{
    "observation": "What implicit political stance does each article take? Through what subtle cues — verb choice, what's foregrounded vs backgrounded, what's questioned vs taken for granted? Note that NYT has its own political position; describe it rather than treating it as neutral baseline.",
    "B_more_or_less_NYT_like": "more|less|similar"
  }},
  "what_B_is_missing": {{
    "observation": "What specifically is in Article A that Article B does not have? Concrete details, specific actors, particular kinds of context, narrative beats, etc."
  }},
  "what_B_does_differently": {{
    "observation": "Where Article B does have content that A doesn't, what fills the space? Generic framings, abstract analysis, generic source attributions, etc."
  }},
  "overall_summary": "One paragraph synthesizing the key differences. What is the single biggest gap between B and A?"
}}

Output ONLY the JSON. No preamble, no markdown fence.
"""

In [ ]:
# =============================================================================
# CELL 4: Run qualitative coding on all 20 pairs
# =============================================================================

CODING_FILE = RESULTS / "qualitative_coding_raw.csv"

# Resume if previously run
if CODING_FILE.exists():
    done_df = pd.read_csv(CODING_FILE)
    done_sids = set(done_df["sample_id"])
    rows = done_df.to_dict("records")
    print(f"Resuming: {len(done_sids)} pairs already coded.")
else:
    rows = []
    done_sids = set()

print(f"To do: {len(pairs_df) - len(done_sids)} pairs")

for _, pair in pairs_df.iterrows():
    sid = pair["sample_id"]
    if sid in done_sids:
        continue

    print(f"\n[{sid}] {pair['news_desk']:<8s}  {pair['headline'][:70]}...")

    user_msg = JUDGE_USER_TEMPLATE.format(
        desk=pair["news_desk"],
        date=pair["pub_date"][:10],
        headline=pair["headline"],
        nyt_text=pair["nyt_text"],
        gpt_text=pair["gpt_text"],
    )

    try:
        resp = client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[
                {"role": "system", "content": JUDGE_SYSTEM},
                {"role": "user", "content": user_msg},
            ],
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS,
            response_format={"type": "json_object"},
        )
        raw_text = resp.choices[0].message.content
        parsed = json.loads(raw_text)
        print(f"    → SUCCESS (response {len(raw_text)} chars)")
    except Exception as e:
        print(f"  !! {type(e).__name__}: {e}")
        parsed = {"error": f"{type(e).__name__}: {e}"}
        raw_text = ""

    # Flatten the JSON for CSV storage
    row = {
        "sample_id": sid,
        "headline": pair["headline"],
        "news_desk": pair["news_desk"],
        "pub_date": pair["pub_date"],
        "writing_register_obs": parsed.get("writing_register", {}).get("observation", ""),
        "writing_register_judgment": parsed.get("writing_register", {}).get("B_more_or_less_NYT_like", ""),
        "stance_framing_obs": parsed.get("stance_and_framing", {}).get("observation", ""),
        "stance_framing_judgment": parsed.get("stance_and_framing", {}).get("B_more_or_less_NYT_like", ""),
        "sociological_obs": parsed.get("sociological_texture", {}).get("observation", ""),
        "sociological_judgment": parsed.get("sociological_texture", {}).get("B_more_or_less_NYT_like", ""),
        "political_slant_obs": parsed.get("political_slant_cues", {}).get("observation", ""),
        "political_slant_judgment": parsed.get("political_slant_cues", {}).get("B_more_or_less_NYT_like", ""),
        "what_B_missing": parsed.get("what_B_is_missing", {}).get("observation", ""),
        "what_B_different": parsed.get("what_B_does_differently", {}).get("observation", ""),
        "overall_summary": parsed.get("overall_summary", ""),
        "raw_response": raw_text,
    }
    rows.append(row)

    # Save after each call (cheap insurance)
    pd.DataFrame(rows).to_csv(CODING_FILE, index=False)
    time.sleep(0.5)

coding_df = pd.DataFrame(rows)
print(f"\n✓ Coded {len(coding_df)} pairs")
print(f"✓ Saved: {CODING_FILE}")


In [ ]:

# =============================================================================
# CELL 5: Quick aggregate — judgment distribution
# =============================================================================

print("=" * 78)
print("JUDGMENT DISTRIBUTION — how often is B (AI-generated) more/less/similar to A (NYT)?")
print("=" * 78)
print()

dimensions = ["writing_register", "stance_framing", "sociological", "political_slant"]
for dim in dimensions:
    col = f"{dim}_judgment"
    if col in coding_df.columns:
        counts = coding_df[col].value_counts()
        print(f"\n{dim.upper().replace('_', ' ')}:")
        for label, count in counts.items():
            print(f"  {label:<15s}  {count:>3d}")

In [ ]:

# =============================================================================
# CELL 6: Print all overall_summary for human reading
# =============================================================================

print("\n" + "=" * 78)
print("OVERALL SUMMARIES (read these to identify themes for paper)")
print("=" * 78)
for _, r in coding_df.sort_values(["news_desk", "sample_id"]).iterrows():
    print(f"\n[{r['news_desk']}] sid={r['sample_id']}: {r['headline'][:70]}")
    print(f"  {r['overall_summary']}")

In [ ]:

# =============================================================================
# CELL 7: Theme extraction — second-pass GPT-4o synthesis
# Read all 20 summaries, identify recurring themes
# =============================================================================

print("\n" + "=" * 78)
print("THEME SYNTHESIS — what recurring patterns emerge across 20 pairs?")
print("=" * 78)

# Concatenate all qualitative observations for second-pass synthesis
all_observations = []
for _, r in coding_df.iterrows():
    all_observations.append(f"""
## Pair sid={r['sample_id']} ({r['news_desk']}, "{r['headline'][:60]}...")

Writing register: {r['writing_register_obs']}
Stance/framing: {r['stance_framing_obs']}
Sociological: {r['sociological_obs']}
Political slant: {r['political_slant_obs']}
What B missing: {r['what_B_missing']}
What B does differently: {r['what_B_different']}
Overall: {r['overall_summary']}
""")

synthesis_prompt = f"""Below are 20 paired-article comparisons between NYT articles and AI-generated articles on the same headlines. Each comparison was made by a careful reader, focusing on writing register, framing, sociological texture, and political slant.

Your task: read all 20 comparisons and identify the 5-7 most recurring themes — patterns that appear across many pairs, not idiosyncratic observations from one comparison. For each theme, provide:

1. A short name (3-6 words)
2. A concise description of what the pattern is
3. Approximately how many of the 20 pairs exhibit it
4. The 1-2 most representative quotes/phrases from the source comparisons

Output as a numbered list. Be specific. Avoid generic claims like "AI is more abstract" — instead specify *what kind* of abstraction, *where* it appears, *what it replaces*.

==== START OF 20 PAIRED COMPARISONS ====
{''.join(all_observations)}
==== END ====
"""

print("Sending to GPT-4o for theme synthesis...")
resp = client.chat.completions.create(
    model=JUDGE_MODEL,
    messages=[
        {"role": "system", "content": "You are a sociologist of media doing thematic analysis."},
        {"role": "user", "content": synthesis_prompt},
    ],
    temperature=0.3,
    max_tokens=3000,
)
synthesis = resp.choices[0].message.content
print()
print(synthesis)

with open(RESULTS / "qualitative_coding_themes.md", "w") as f:
    f.write("# Qualitative Coding: Recurring Themes\n\n")
    f.write(synthesis)
print(f"\nSaved themes synthesis to {RESULTS / 'qualitative_coding_themes.md'}")

In [ ]:


# =============================================================================
# CELL 8: Summary
# =============================================================================

print("\n" + "=" * 78)
print("NOTEBOOK 05 COMPLETE")
print("=" * 78)
for f in sorted(RESULTS.glob("qualitative_*")):
    size = f.stat().st_size
    print(f"  {f.name:50s}  {size:>10,} bytes")
print("""
Next steps:
1. Read the themes synthesis (qualitative_coding_themes.md) — these are
   the patterns to incorporate into Section 5 (Discussion) of the paper.
2. Optionally, hand-verify the most important pair (e.g., panda story or
   Hong Kong arrests) to ensure GPT-4o's coding matches your reading.
3. Use specific quotes from `qualitative_coding_raw.csv` when writing
   Discussion paragraphs.
""")